In [29]:
from re import TEMPLATE
import sympy as sp
import itertools
import numpy as np

def even_combinations(k):
    n = k-1
    if n%2 == 0 or n < 0:
        return  # No solutions for odd or negative n
    for i in range(1,n+1,2):
      for r in range(0, n+1-i, 1):
          for combination in itertools.combinations_with_replacement(range(2, n + 1-i, 2), r):
              if sum(combination)+i == n:
                  combination = (i,) + combination
                  yield combination




#getting expression
def getExpression(n):
  tauB,gamma0,t,t1,t2,t3,t4,t5 = sp.symbols('tauB gamma0 t t1 t2 t3 t4 t5', real = True, positive = True)
  timeDict = {0:t,1:t1,2:t2,3:t3,4:t4,5:t5}
  def correlationFunction(i,j):
    return gamma0/(2*tauB)*sp.exp(-1/tauB*(timeDict[i]-timeDict[j]))

  fullComboList = []
  def getCombos(combo,curr,L_arr,comboidx = 0): #recursive function with args combo: even_combinations; curr: filled array to ret; L_arr: L's to use
    if L_arr == []:
        result = curr.copy()
        fullComboList.append(result)
        return
    for currCombo in itertools.combinations(L_arr,combo[comboidx]):
        temp_L_arr = L_arr.copy()
        temp_curr = curr.copy()
        for item in currCombo:
            temp_L_arr.remove(item)
        temp_curr.append(sorted(list(currCombo)))
        getCombos(combo,temp_curr,temp_L_arr,comboidx+1)

  L_arr = [i for i in range(1,n)]
  for combination in even_combinations(n):
      getCombos(combination,[],L_arr)

  fullExpr = 0
  for term in fullComboList:
    q = len(term)
    flattened = [val for sublist in term for val in sublist]
    flattened.insert(0,0)
    currTerm = 1
    for i,j in zip(flattened[::2],flattened[1::2]):
      currTerm *= -(-1)**q*correlationFunction(i,j)
    fullExpr += currTerm

  sp.simplify(fullExpr)
  #now, with full expr, time to integrate
  result = fullExpr
  L_arr.insert(0,0)
  for i,j in zip(L_arr[:0:-1],L_arr[::-1][1:]):
      result = sp.integrate(
        result,
        (timeDict[i], 0, timeDict[j]),
      )

  return sp.simplify(result)



print(getExpression(6))



gamma0**3*(tauB*(-4*t*exp(t/tauB) + tauB*(2*exp(3*t/tauB) - 1) - 2*tauB*exp(t/tauB)) + (-2*t**2 - 2*t*tauB + tauB**2)*exp(2*t/tauB))*exp(-3*t/tauB)/8
